# Assignment Template

Please follow the template below to submit part A of your assignment. Make sure all the outputs are provided within the submission file. Make sure to familarise yourself with the assignment brief before.

# Libraries

In [2]:
!pip install transformers datasets rouge_score -q

  Preparing metadata (setup.py) ... done


In [3]:
from transformers import pipeline
from datasets import load_dataset
from rouge_score import rouge_scorer
import pandas as pd

# All Pre-requsites (Hugging Face login, data import etc.)

Note: If you use a HuggingFace model, please do not share your access key, make sure all the outputs are visible, you will be marked down if outputs are not provided)

Provide all functions that you might need for processing the data and using the model (refer to Unit 4 tutorial for some hints)

In [4]:
dataset = load_dataset("EdinburghNLP/xsum")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})


In [5]:
test_sample = dataset["test"].shuffle(seed=42).select(range(100))
print(f"Sample size: {len(test_sample)}")
print(f"Example article: {test_sample[0]['document'][:400]}")
print(f"Example summary: {test_sample[0]['summary']}")

Sample size: 100
Example article: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charit
Example summary: A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.


In [6]:
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = 'facebook/bart-large-cnn'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to("cuda")
print("Model loaded successfully.")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully.


In [7]:
def evaluate_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        r1.append(score["rouge1"].fmeasure)
        r2.append(score["rouge2"].fmeasure)
        rl.append(score["rougeL"].fmeasure)
    return {
        "ROUGE-1": round(sum(r1)/len(r1), 4),
        "ROUGE-2": round(sum(r2)/len(r2), 4),
        "ROUGE-L": round(sum(rl)/len(rl), 4)
    }

# Zero-shot Strategy

In [8]:
def zero_shot_summarize(article):
    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=10,
        length_penalty=2.0,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [9]:
zero_shot_predictions = []

for i, example in enumerate(test_sample):
    summary = zero_shot_summarize(example["document"])
    zero_shot_predictions.append(summary)
    if i % 10 == 0:
        print(f"Processed {i}/100")

print("Zero-shot complete!")

Processed 0/100
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Zero-shot complete!


In [29]:
for i in range(3):
  print(f"Article: {test_sample[i]['document']}\n\n")
  print(f"Reference: {test_sample[i]['summary']}\n\n")
  print(f"Generated: {zero_shot_predictions[i]}\n\n")
  print("---------------------------------------------------------------------------------------------------------")

Article: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she was there she got su

# Few-shot Strategy

In [11]:
def few_shot_summarize(article):
    examples = dataset["train"].select([0, 1])

    prompt = ""
    for ex in examples:
        prompt += f"Article: {ex['document'][:500]}\nSummary: {ex['summary']}\n\n"
    prompt += f"Article: {article[:500]}\nSummary:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=10,
        length_penalty=2.0,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [12]:
few_shot_predictions = []

for i, example in enumerate(test_sample):
    summary = few_shot_summarize(example["document"])
    few_shot_predictions.append(summary)
    if i % 10 == 0:
        print(f"Processed {i}/100")

print("Few-shot complete!")

Processed 0/100
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Few-shot complete!


In [25]:
for i in range(3):
    print(f"Article: {test_sample[i]['document']}\n\n")
    print(f"Reference: {test_sample[i]['summary']}\n\n")
    print(f"Generated: {few_shot_predictions[i]}\n\n")
    print("---------------------------------------------------------------------------------------------------")

Article: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she was there she got su

# Prompt Engineering

## Zero-shot

In [14]:
def prompt_eng_zero_shot(article):
    prompt = (
        "Summarise the following BBC news article in one concise sentence. "
        "Focus on the most important fact. Do not include unnecessary detail.\n\n"
        f"Article: {article[:800]}\nSummary:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=10,
        length_penalty=2.0,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [15]:
pe_zero_shot_predictions = []

for i, example in enumerate(test_sample):
    summary = prompt_eng_zero_shot(example["document"])
    pe_zero_shot_predictions.append(summary)
    if i % 10 == 0:
        print(f"Processed {i}/100")

print("Prompt engineering zero-shot complete!")

Processed 0/100
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Prompt engineering zero-shot complete!


In [30]:
for i in range(3):
    print(f"Article: {test_sample[i]['document']}\n\n")
    print(f"Reference: {test_sample[i]['summary']}\n\n")
    print(f"Generated: {pe_zero_shot_predictions[i]}\n\n")
    print("------------------------------------------------------------------------------")

Article: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she was there she got su

## Few-shot

In [17]:
def prompt_eng_few_shot(article):
    examples = dataset["train"].select([0, 1])

    prompt = (
        "Summarise the following BBC news article in one concise sentence. "
        "Focus on the most important fact.\n\n"
    )
    for ex in examples:
        prompt += f"Article: {ex['document'][:400]}\nSummary: {ex['summary']}\n\n"
    prompt += f"Article: {article[:400]}\nSummary:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=10,
        length_penalty=2.0,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [18]:
pe_few_shot_predictions = []

for i, example in enumerate(test_sample):
    summary = prompt_eng_few_shot(example["document"])
    pe_few_shot_predictions.append(summary)
    if i % 10 == 0:
        print(f"Processed {i}/100")

print("Prompt engineering few-shot complete!")

Processed 0/100
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Prompt engineering few-shot complete!


In [31]:
for i in range(3):
    print(f"Article: {test_sample[i]['document']}\n\n")
    print(f"Reference: {test_sample[i]['summary']}\n\n")
    print(f"Generated: {pe_few_shot_predictions[i]}\n\n")
    print("---------------------------------------------------------------------------------------")

Article: Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
Minibus driver James Johnson was jailed for more than six years for causing Bethany's death, in April 2013.
Ms Johnson, who broke her shoulder, back and pelvis, said the help she received from a charity while in hospital led her to want to support others.
Speaking publicly for the first time about the crash, Ms Johnson described how everyone was "excited and giddy" for the hen party.
"To me the impact was just a massive explosion," she said.  "I thought the bus had blown up.
"I remember the bus dropping on its side. The next thing, I woke up on the roadside so I'd actually come out of the window."
Ms Johnson was taken to Leeds General Infirmary where she, along with Bethany's sister Amy Firth, underwent major surgery and spent time in intensive care.
Whilst she was there she got su

# Evaluation

In [ ]:
references = [example["summary"] for example in test_sample]

zero_shot_scores = evaluate_rouge(zero_shot_predictions, references)
few_shot_scores = evaluate_rouge(few_shot_predictions, references)
pe_zero_scores = evaluate_rouge(pe_zero_shot_predictions, references)
pe_few_scores = evaluate_rouge(pe_few_shot_predictions, references)

print("Zero-shot:", zero_shot_scores)
print("Few-shot:", few_shot_scores)
print("PE Zero-shot:", pe_zero_scores)
print("PE Few-shot:", pe_few_scores)

Zero-shot: {'ROUGE-1': 0.2183, 'ROUGE-2': 0.0374, 'ROUGE-L': 0.1508}
Few-shot: {'ROUGE-1': 0.0983, 'ROUGE-2': 0.0041, 'ROUGE-L': 0.0796}
PE Zero-shot: {'ROUGE-1': 0.1998, 'ROUGE-2': 0.0223, 'ROUGE-L': 0.1352}
PE Few-shot: {'ROUGE-1': 0.1055, 'ROUGE-2': 0.0073, 'ROUGE-L': 0.0815}


In [ ]:
results = pd.DataFrame({
    "Approach": [
        "Zero-shot",
        "Few-shot",
        "Prompt Engineering (Zero-shot)",
        "Prompt Engineering (Few-shot)"
    ],
    "ROUGE-1": [
        zero_shot_scores["ROUGE-1"],
        few_shot_scores["ROUGE-1"],
        pe_zero_scores["ROUGE-1"],
        pe_few_scores["ROUGE-1"]
    ],
    "ROUGE-2": [
        zero_shot_scores["ROUGE-2"],
        few_shot_scores["ROUGE-2"],
        pe_zero_scores["ROUGE-2"],
        pe_few_scores["ROUGE-2"]
    ],
    "ROUGE-L": [
        zero_shot_scores["ROUGE-L"],
        few_shot_scores["ROUGE-L"],
        pe_zero_scores["ROUGE-L"],
        pe_few_scores["ROUGE-L"]
    ]
})

print(results.to_string(index=False))

                      Approach  ROUGE-1  ROUGE-2  ROUGE-L
                     Zero-shot   0.2183   0.0374   0.1508
                      Few-shot   0.0983   0.0041   0.0796
Prompt Engineering (Zero-shot)   0.1998   0.0223   0.1352
 Prompt Engineering (Few-shot)   0.1055   0.0073   0.0815


In [ ]:
results.style.highlight_max(axis=0, props="color: black; background-color: lightgreen")

,Approach,ROUGE-1,ROUGE-2,ROUGE-L
0,Zero-shot,0.218300,0.037400,0.150800
1,Few-shot,0.098300,0.004100,0.079600
2,Prompt Engineering (Zero-shot),0.199800,0.022300,0.135200
3,Prompt Engineering (Few-shot),0.105500,0.007300,0.081500


# Results Interpretation (max 200 words)

Note: If you reference something from the model outputs, make sure it can be traced (e.g., if you noticed a interesting pattern in the results, make sure to provide an example of that in the outputs).




I have evaluated and scored the four approaches on 100 XSum test examples using facebook/bart-large-cnn and ROUGE-1, ROUGE-2 and ROUGE-L metrics.

**FINDINGS:** The zero shot approach performed best across all metrics (ROUGE-1: 0.2183). This suggests that the model's summarisation training already aligns well this task and providing extra guidance in the form of prompts did not imrpove it's performance. However, Prompt Engineering Zero-shot's close score (ROUGE-1: 0.1998) indicates that explicit instructions provided little benefit over none at all.

**SURPRISING ERRORS:** The most surprising error was the significant drop in both few-shot approaches. Few shot scored only 0.0983 on ROUGE-1, which is less than half of the zero-shot score for the same. After carefully reviewing the output, it is clear that the model was not designed to follow conversation prompt structures. Unlike models like GPT which are more tuned to following instructions, BART is an encoder-decoder model that is finetuned mainly for summarisation. As a result, the few-shot approach seems to be ineffective; instead of using the examples as demonstrations, it summarises the whole prompt as a single document and produces output completely unrelated to the article. For example, the article about the car crash received a summary about flooding in Newton Stewart. In addition, BART also ignores the constraints provided by the prompts, like when it was prompted to limit the summary to one concise sentence (in the prompt engineering zero-shot article about Braham Hall Stockport). This further demonstrates that BART responds to generation parameters rather than natural language processing. This shows that the few-shot approach and prompting are best suited to instruction-following models like GPT rather than task-specific ones like BART.

**BENCHMARK VS ACTUAL PERFORMANCE:** BART-large-CNN is reported to
achieve ROUGE-1 scores around 0.43 on the CNN/DailyMail dataset which is approximately double of the 0.22 I got. This is expected since XSum summaries are more abstractive and harder to match exactly. Also, the benchmark used full fine-tuning on the dataset, whereas this evaluation is basically just zero-shot.